# Reconstruction de contexte et sous-graphes

Ce notebook part d'un export d'événements de type PREVYO / EMVISTA, puis :

1. normalise les champs Mongo (`$oid`, `$date`)
2. reconstruit les **sous-graphes locaux** de chaque événement
3. construit un **graphe global de contexte**
4. affiche un **ego-graph** autour d'un événement
5. donne une base pour explorer `type`, `domain`, `taskId`, `risk`, `sourceNodeId`


In [32]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

from IPython.display import display, Markdown

In [22]:
DATA_PATH = Path("data/export.events.json")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_events = json.load(f)

len(raw_events)

11948

In [23]:
def unwrap_mongo(value):
    if isinstance(value, dict):
        if "$oid" in value:
            return value["$oid"]
        if "$date" in value:
            return value["$date"]
        return {k: unwrap_mongo(v) for k, v in value.items()}
    if isinstance(value, list):
        return [unwrap_mongo(v) for v in value]
    return value

events = [unwrap_mongo(event) for event in raw_events]
events[:1]

[{'_id': '698b35e78f03efde04a87998',
  'createdAt': '2026-02-10T13:43:03.827Z',
  'type': 'Thing/Abstract/Event/Win',
  'subdomain': 'Thing/Abstract/Domain/Digital',
  'risk': 'Thing/Abstract/Risk/Societal',
  'riskCarac': 'Thing/Abstract/Domain/Sovereignty',
  'sourceNodeId': '474738074',
  'context': 'Portrait des six principaux candidats qui sont en lice pour remporter le fauteuil de maire de la cité phocéenne à l’issue des élections municipales des 15et 22mars.',
  'resultAnalyseId': '698b35e042a8083a290c11c7',
  'taskId': '1a076139-ea5e-40e2-8d82-01a1f28b8181',
  'nodes': [{'_id': '474738074',
    'form': 'remporter',
    'labels': ['Thing/Abstract/Event/Win'],
    'properties': {'mood': 'INF',
     'aspect': 'STATE',
     'category': 'GENERAL',
     'tense': 'PRES',
     'polarity': 'POS'}},
   {'_id': '2080660730',
    'form': 'fauteuil',
    'labels': ['Thing/Concrete/Inanimate'],
    'properties': {'quantity:exact': '1'}},
   {'_id': '1212458420',
    'form': '15et 22mars',
  

On applatie les événements en tables

On fabrique trois niveaux :

- `events_df` : 1 ligne par événement
- `nodes_df` : 1 ligne par nœud détecté
- `edges_df` : 1 ligne par relation

In [24]:
event_rows = []
node_rows = []
edge_rows = []

for event in events:
    event_id = event.get("_id")

    event_rows.append({
        "event_id": event_id,
        "createdAt": event.get("createdAt"),
        "endAt": event.get("endAt"),
        "type": event.get("type"),
        "domain": event.get("domain"),
        "subdomain": event.get("subdomain"),
        "risk": event.get("risk"),
        "riskCarac": event.get("riskCarac"),
        "sourceNodeId": event.get("sourceNodeId"),
        "taskId": event.get("taskId"),
        "resultAnalyseId": event.get("resultAnalyseId"),
        "context": event.get("context"),
        "n_nodes": len(event.get("nodes", [])),
        "n_edges": len(event.get("edges", [])),
    })

    for node in event.get("nodes", []):
        node_rows.append({
            "event_id": event_id,
            "node_id": node.get("_id"),
            "form": node.get("form"),
            "labels": tuple(node.get("labels", [])),
            "main_label": (node.get("labels") or [None])[0],
            "properties": node.get("properties", {}),
        })

    for edge in event.get("edges", []):
        edge_rows.append({
            "event_id": event_id,
            "edge_id": edge.get("_id"),
            "edge_type": edge.get("type"),
            "source": edge.get("source"),
            "target": edge.get("target"),
            "properties": edge.get("properties", {}),
        })

events_df = pd.DataFrame(event_rows)
nodes_df = pd.DataFrame(node_rows)
edges_df = pd.DataFrame(edge_rows)

display(events_df)
display(nodes_df.head(10))
display(edges_df.head(10))

,event_id,createdAt,endAt,type,domain,subdomain,risk,riskCarac,sourceNodeId,taskId,resultAnalyseId,context,n_nodes,n_edges
0,698b35e78f03efde04a87998,2026-02-10T13:43:03.827Z,None,Thing/Abstract/Event/Win,NaN,Thing/Abstract/Domain/Digital,Thing/Abstract/Risk/Societal,Thing/Abstract/Domain/Sovereignty,474738074,1a076139-ea5e-40e2-8d82-01a1f28b8181,698b35e042a8083a290c11c7,Portrait des six principaux candidats qui sont...,3,2
1,698b35e98f03efde04a8799a,2026-02-10T13:43:05.544Z,2026-02-10T13:42:59.932Z,Thing/Abstract/Event/Transfer/TransferOfUnbias...,Thing/Abstract/Event/Spying,NaN,NaN,NaN,1667751063,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,698b35e042a8083a290c11c5,Des e-mails publiés par le département américa...,3,2
2,698b35e98f03efde04a8799b,2026-02-10T13:43:05.544Z,None,Thing/Abstract/Event/Transfer/TransferOfUnbias...,Thing/Abstract/Event/Spying,NaN,NaN,NaN,562551427,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,698b35e042a8083a290c11c5,Des e-mails publiés par le département américa...,3,2
3,698b35e98f03efde04a8799c,2026-02-10T13:43:05.544Z,None,Thing/Abstract/Event/Build,Thing/Abstract/Event/Spying,NaN,NaN,NaN,397183198,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,698b35e042a8083a290c11c5,ervi de facilitateur à des représentants de l’...,1,0
4,698b35e98f03efde04a8799d,2026-02-10T13:43:05.544Z,None,Thing/Abstract/Event/Addition,Thing/Abstract/Event/Spying,NaN,NaN,NaN,722640268,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,698b35e042a8083a290c11c5,Des e-mails publiés par le département américa...,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11943,699ea4e5ad1db2119a5fae34,2026-02-25T07:29:41.380Z,None,Thing/Abstract/Event,NaN,NaN,NaN,NaN,1591273328,72cc0f71-c467-416c-8b29-bfc48f61eab7,699ea4d94469fb97fca64fcc,appuyé au président Lula. Candidat à sa succes...,1,0
11944,699ea4e5ad1db2119a5fae35,2026-02-25T07:29:41.380Z,None,Thing/Abstract/Event/Begin,NaN,NaN,NaN,NaN,1116254576,72cc0f71-c467-416c-8b29-bfc48f61eab7,699ea4d94469fb97fca64fcc,ernier est accusé d’avoir instrumentalisé l’év...,2,1
11945,699ea4e5ad1db2119a5fae36,2026-02-25T07:29:41.380Z,None,Thing/Abstract/Event,NaN,NaN,NaN,NaN,558383439,72cc0f71-c467-416c-8b29-bfc48f61eab7,699ea4d94469fb97fca64fcc,voir instrumentalisé l’événement pour faire de...,1,0
11946,699ea737ad1db2119a5fae38,2026-02-25T07:39:35.211Z,2026-02-25T07:39:29.218Z,Thing/Abstract/Event/Want,NaN,NaN,NaN,NaN,1474411858,dd498b42-bdce-4efd-b92d-2df817c9e0f4,699ea7314469fb97fca64fcd,"Attendu à Kinshasa pour un combat inédit, Tony...",4,3


,event_id,node_id,form,labels,main_label,properties
0,698b35e78f03efde04a87998,474738074,remporter,"(Thing/Abstract/Event/Win,)",Thing/Abstract/Event/Win,"{'mood': 'INF', 'aspect': 'STATE', 'category':..."
1,698b35e78f03efde04a87998,2080660730,fauteuil,"(Thing/Concrete/Inanimate,)",Thing/Concrete/Inanimate,{'quantity:exact': '1'}
2,698b35e78f03efde04a87998,1212458420,15et 22mars,"(Thing/Abstract/Time,)",Thing/Abstract/Time,{}
3,698b35e98f03efde04a8799a,1667751063,publiés,(Thing/Abstract/Event/Transfer/TransferOfUnbia...,Thing/Abstract/Event/Transfer/TransferOfUnbias...,"{'mood': 'PART', 'aspect': 'STATE', 'category'..."
4,698b35e98f03efde04a8799a,133445445,2026-02-10T13:42:59.932651238,"(Thing/Abstract/Time,)",Thing/Abstract/Time,"{'timestamp': [2026, 2, 10, 13, 42, 59, 932651..."
5,698b35e98f03efde04a8799a,1664342503,e-mails,(Thing/Concrete/Inanimate/Product/Communicatio...,Thing/Concrete/Inanimate/Product/Communication...,{'quantity:min': '2'}
6,698b35e98f03efde04a8799b,562551427,témoignent,(Thing/Abstract/Event/Transfer/TransferOfUnbia...,Thing/Abstract/Event/Transfer/TransferOfUnbias...,"{'mood': 'IND', 'aspect': 'ATELIC_PROCESS', 'c..."
7,698b35e98f03efde04a8799b,1667751063,publiés,(Thing/Abstract/Event/Transfer/TransferOfUnbia...,Thing/Abstract/Event/Transfer/TransferOfUnbias...,"{'mood': 'PART', 'aspect': 'STATE', 'category'..."
8,698b35e98f03efde04a8799b,310862638,servi,"(Thing,)",Thing,{}
9,698b35e98f03efde04a8799c,397183198,montages,"(Thing/Abstract/Event/Build,)",Thing/Abstract/Event/Build,"{'mood': 'EMPTY', 'aspect': 'PROCESS', 'catego..."


,event_id,edge_id,edge_type,source,target,properties
0,698b35e78f03efde04a87998,0,Stimulus,474738074,2080660730,{}
1,698b35e78f03efde04a87998,1,Time,474738074,1212458420,{}
2,698b35e98f03efde04a8799a,0,TimeMax,1667751063,133445445,{}
3,698b35e98f03efde04a8799a,1,Theme,1667751063,1664342503,{}
4,698b35e98f03efde04a8799b,2,TimeMin,562551427,1667751063,{}
5,698b35e98f03efde04a8799b,3,Topic,562551427,310862638,{}
6,698b35e98f03efde04a8799d,9,Addition,722640268,886966870,{}
7,698b35e98f03efde04a8799d,10,Addition,722640268,824289104,{}
8,698b35ea8f03efde04a8799f,0,TimeMax,1413780907,153927880,{}
9,698b35ea8f03efde04a8799f,1,Agent,1413780907,1089471328,{}


In [25]:
summary = {
    "n_events": len(events_df),
    "n_nodes_total": len(nodes_df),
    "n_edges_total": len(edges_df),
    "types": events_df["type"].value_counts(dropna=False).to_dict(),
    "domains": events_df["domain"].fillna("NA").value_counts().to_dict(),
    "tasks": events_df["taskId"].value_counts(dropna=False).to_dict(),
}

# summary 

On construit les sous-graphes locaux

Chaque événement devient un petit graphe orienté basé sur ses `nodes` et `edges`.

In [26]:
def build_event_subgraph(event):
    G = nx.DiGraph()
    event_id = event["_id"]

    for node in event.get("nodes", []):
        G.add_node(
            node["_id"],
            event_id=event_id,
            label=node.get("form"),
            main_label=(node.get("labels") or [None])[0],
            labels=node.get("labels", []),
            node_type="semantic_node",
            properties=node.get("properties", {}),
        )

    for edge in event.get("edges", []):
        if edge.get("source") in G.nodes and edge.get("target") in G.nodes:
            G.add_edge(
                edge["source"],
                edge["target"],
                edge_type=edge.get("type"),
                event_id=event_id,
                properties=edge.get("properties", {}),
            )

    return G

subgraphs = {event["_id"]: build_event_subgraph(event) for event in events}
{k: (g.number_of_nodes(), g.number_of_edges()) for k, g in subgraphs.items()}

{'698b35e78f03efde04a87998': (3, 2),
 '698b35e98f03efde04a8799a': (3, 2),
 '698b35e98f03efde04a8799b': (3, 2),
 '698b35e98f03efde04a8799c': (1, 0),
 '698b35e98f03efde04a8799d': (3, 2),
 '698b35ea8f03efde04a8799f': (6, 5),
 '698b35ea8f03efde04a879a0': (2, 1),
 '698b35ea8f03efde04a879a1': (4, 3),
 '698b35ea8f03efde04a879a2': (1, 0),
 '698b35ea8f03efde04a879a3': (1, 0),
 '698b35ea8f03efde04a879a4': (5, 4),
 '698b35ed8f03efde04a879a6': (1, 0),
 '698b35ed8f03efde04a879a7': (5, 4),
 '698b35ed8f03efde04a879a8': (1, 0),
 '698b35ed8f03efde04a879a9': (4, 3),
 '698b35ed8f03efde04a879aa': (3, 2),
 '698b35ed8f03efde04a879ac': (1, 0),
 '698b35ed8f03efde04a879ad': (4, 3),
 '698b35ed8f03efde04a879ae': (2, 1),
 '698b35ed8f03efde04a879af': (5, 4),
 '698b35ed8f03efde04a879b1': (6, 5),
 '698b35ed8f03efde04a879b2': (1, 0),
 '698b35ed8f03efde04a879b3': (1, 0),
 '698b35ed8f03efde04a879b4': (3, 2),
 '698b35ef8f03efde04a879b6': (3, 2),
 '698b35ef8f03efde04a879b7': (1, 0),
 '698b35ef8f03efde04a879b8': (4, 3),
 

On utilise Plotly pour avoir une visualisation interactive.

In [27]:
def color_for_node(main_label):
    if not main_label:
        return "#9ca3af"
    if "Time" in main_label:
        return "#f59e0b"
    if "Event" in main_label:
        return "#3b82f6"
    if "CommunicationMedium" in main_label:
        return "#10b981"
    if "Inanimate" in main_label:
        return "#8b5cf6"
    return "#64748b"


def plot_graph(G, title="Graphe", layout_seed=7, width=950, height=650):
    if G.number_of_nodes() == 0:
        print("Graphe vide")
        return

    pos = nx.spring_layout(
        G,
        seed=layout_seed,
        k=1.2 / max(1, (len(G.nodes()) ** 0.5))
    )

    edge_x = []
    edge_y = []
    edge_mid_x = []
    edge_mid_y = []
    edge_text = []

    for u, v, data in G.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
        edge_mid_x.append((x0 + x1) / 2)
        edge_mid_y.append((y0 + y1) / 2)
        edge_text.append(data.get("edge_type", ""))

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(width=1.3, color="rgba(120,120,120,0.55)"),
        hoverinfo="none",
        showlegend=False,
    )

    edge_label_trace = go.Scatter(
        x=edge_mid_x,
        y=edge_mid_y,
        mode="text",
        text=edge_text,
        textposition="middle center",
        hoverinfo="text",
        textfont=dict(size=10),
        showlegend=False,
    )

    node_x = []
    node_y = []
    node_text = []
    node_color = []
    node_size = []

    for node_id, data in G.nodes(data=True):
        x, y = pos[node_id]
        node_x.append(x)
        node_y.append(y)

        label = data.get("label", str(node_id))
        main_label = data.get("main_label", "")
        tooltip = f"id: {node_id}<br>form: {label}<br>label: {main_label}"

        node_text.append(tooltip)
        node_color.append(color_for_node(main_label))

        degree = G.degree(node_id)
        node_size.append(20 + degree * 4)

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=[G.nodes[n].get("label", str(n)) for n in G.nodes()],
        textposition="top center",
        hovertext=node_text,
        hoverinfo="text",
        marker=dict(
            size=node_size,
            color=node_color,
            line=dict(width=1, color="white")
        ),
        showlegend=False,
    )

    fig = go.Figure(data=[edge_trace, edge_label_trace, node_trace])
    fig.update_layout(
        title=title,
        width=width,
        height=height,
        margin=dict(l=20, r=20, t=50, b=20),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        plot_bgcolor="white",
    )
    fig.show()

In [30]:
selected_event_id = events_df.iloc[4]["event_id"]
selected_event_row = events_df.loc[events_df["event_id"] == selected_event_id].iloc[0]

display(selected_event_row.to_frame().rename(columns={selected_event_row.name: "value"}))

plot_graph(
    subgraphs[selected_event_id],
    title=f"Sous-graphe local · {selected_event_id} · {selected_event_row['type']}"
)

,value
event_id,698b35e98f03efde04a8799d
createdAt,2026-02-10T13:43:05.544Z
endAt,None
type,Thing/Abstract/Event/Addition
domain,Thing/Abstract/Event/Spying
subdomain,NaN
risk,NaN
riskCarac,NaN
sourceNodeId,722640268
taskId,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75


In [33]:
subgraph_index = pd.DataFrame([
    {
        "event_id": event_id,
        "n_nodes": G.number_of_nodes(),
        "n_edges": G.number_of_edges(),
        "forms": ", ".join(
            str(G.nodes[n].get("label"))
            for n in list(G.nodes)[:5]
        ),
    }
    for event_id, G in subgraphs.items()
]).sort_values(["n_nodes", "n_edges"], ascending=False)

display(subgraph_index.head(20))

,event_id,n_nodes,n_edges,forms
842,698b64cb8f03efde04a87d97,11,10,"jugés, 0006-01-01T00:00, personne, policiers, ..."
989,698b8a4e8f03efde04a87e45,9,8,"lancé, 2026-02-03T23:59, 2026-02-03T00:00, , m..."
4526,69905ce534efc5d3aac0fe6c,8,8,"intégrer, 2026-02-21T23:59, astronaute Sophie ..."
7901,6998408e6736db9dc26e6066,8,8,"penserez, devenu, pharaon, 2026-02-20T11:07:46..."
10115,699c09a16736db9dc26e6aad,8,8,"examiné, 2026-02-17T23:59, estime, 2026-02-17T..."
10874,699cda736736db9dc26e6e37,8,8,"trouver, est, mise en œuvre, 2026-02-23T22:53:..."
2046,698ccf2a9966a5e0139f0bfe,8,7,"ajouté, 2026-02-04T23:59, 2026-02-04T00:00, dé..."
5065,69919e5e34efc5d3aac100f5,8,7,"et, envole, démarre, fait, enflamme"
5120,6991ac5e34efc5d3aac10139,8,7,"partent, 2026-02-22T23:59, match, 2026-02-22T0..."
5753,6992f99534efc5d3aac1042f,8,7,"rend, chef, prévues, en difficulté, soutien"


In [36]:
for event_id in subgraph_index["event_id"].head(1):
    row = events_df.loc[events_df["event_id"] == event_id].iloc[0]

    print("=" * 110)
    print("event_id :", event_id)
    print("type     :", row["type"])
    print("domain   :", row["domain"])
    print("taskId   :", row["taskId"])
    print("context  :", row["context"])

    plot_graph(subgraphs[event_id], title=f"Sous-graphe · {event_id}")

event_id : 698b64cb8f03efde04a87d97
type     : Thing/Abstract/Event/Judgment
domain   : nan
taskId   : 10aeddf2-c647-4fd3-bb06-e5b981915087
context  : Six ans après les faits, trois policiers seront jugés, du 9 au 19novembre, notamment pour violences volontaires avec incapacité totale de travail supérie


In [37]:
def build_global_context_graph(events):
    G = nx.DiGraph()

    for event in events:
        event_id = event["_id"]
        event_node = f"event::{event_id}"

        G.add_node(
            event_node,
            node_type="event_anchor",
            label=event.get("type", "event"),
            event_id=event_id,
            type=event.get("type"),
            domain=event.get("domain"),
            taskId=event.get("taskId"),
            context=event.get("context"),
        )

        for node in event.get("nodes", []):
            node_id = node["_id"]

            if node_id not in G:
                G.add_node(
                    node_id,
                    node_type="semantic_node",
                    label=node.get("form"),
                    main_label=(node.get("labels") or [None])[0],
                    labels=node.get("labels", []),
                    properties=node.get("properties", {}),
                )
            else:
                existing_events = G.nodes[node_id].get("event_ids", set())
                if not isinstance(existing_events, set):
                    existing_events = set(existing_events)
                existing_events.add(event_id)
                G.nodes[node_id]["event_ids"] = existing_events

            G.add_edge(event_node, node_id, edge_type="contains")

            if node_id == event.get("sourceNodeId"):
                G.add_edge(event_node, node_id, edge_type="sourceNode")

        for edge in event.get("edges", []):
            src = edge.get("source")
            tgt = edge.get("target")

            if src in G.nodes and tgt in G.nodes:
                G.add_edge(
                    src,
                    tgt,
                    edge_type=edge.get("type"),
                    event_id=event_id
                )

    return G

G_global = build_global_context_graph(events)
print("Nb nodes :", G_global.number_of_nodes())
print("Nb edges :", G_global.number_of_edges())

Nb nodes : 41684
Nb edges : 59794


In [38]:
node_type_counts = Counter(nx.get_node_attributes(G_global, "node_type").values())
edge_type_counts = Counter(nx.get_edge_attributes(G_global, "edge_type").values())

print("Node types:", dict(node_type_counts))
print("Edge types:", dict(edge_type_counts))

Node types: {'event_anchor': 11948, 'semantic_node': 29736}
Edge types: {'sourceNode': 11948, 'contains': 23754, 'Stimulus': 114, 'Time': 863, 'TimeMax': 3092, 'Theme': 3973, 'TimeMin': 2126, 'Topic': 591, 'Addition': 2044, 'Agent': 2114, 'Purpose': 433, 'Recipient': 309, 'Location': 2464, 'TimeExact': 1807, 'Beneficiary': 116, 'Instrument': 273, 'Patient': 464, 'Cause': 274, 'Illustration': 28, 'Result': 49, 'Pivot': 468, 'ArgumentOut': 354, 'ArgumentIn': 354, 'Manner': 268, 'Experiencer': 187, 'Value': 65, 'Destination': 99, 'Event': 58, 'Restriction': 36, 'Opposition': 203, 'Attribute': 137, 'Alternative': 169, 'Source': 170, 'Product': 137, 'Material': 18, 'Comparison': 62, 'Consequence': 47, 'TimeDuration': 16, 'Asset': 33, 'Enumeration': 6, 'LocationExact': 14, 'Condition': 34, 'Initial_Location': 6, 'Whatever': 1, 'LocationSource': 1, 'IL': 8, 'Explanation': 1, 'Extent': 3, 'Conclusion': 2, 'Measure': 1}


In [39]:
selected_event_id = events_df.iloc[0]["event_id"]
anchor = f"event::{selected_event_id}"

ego = nx.ego_graph(G_global, anchor, radius=2, undirected=False)
plot_graph(ego, title=f"Ego-graph de contexte · {selected_event_id}")

In [41]:
FILTER_DOMAIN = "Thing/Abstract/Event/Spying"

filtered_events = [e for e in events if e.get("domain") == FILTER_DOMAIN]
filtered_graph = build_global_context_graph(filtered_events)

print("Nombre d'événements filtrés :", len(filtered_events))
print(
    "Taille du graphe filtré :",
    filtered_graph.number_of_nodes(),
    "nœuds /",
    filtered_graph.number_of_edges(),
    "arêtes"
)

if filtered_graph.number_of_nodes() > 0:
    plot_graph(filtered_graph, title=f"Graphe global filtré domain = {FILTER_DOMAIN}")
else:
    print("Aucun événement pour ce domain")

Nombre d'événements filtrés : 263
Taille du graphe filtré : 939 nœuds / 1369 arêtes


In [42]:
FILTER_TASK_ID = events_df["taskId"].dropna().iloc[0]

filtered_events_task = [e for e in events if e.get("taskId") == FILTER_TASK_ID]
filtered_graph_task = build_global_context_graph(filtered_events_task)

print("TaskId sélectionné :", FILTER_TASK_ID)
print("Nombre d'événements :", len(filtered_events_task))

if filtered_graph_task.number_of_nodes() > 0:
    plot_graph(filtered_graph_task, title=f"Graphe global filtré · taskId = {FILTER_TASK_ID}")

TaskId sélectionné : 1a076139-ea5e-40e2-8d82-01a1f28b8181
Nombre d'événements : 1


In [43]:
FILTER_TYPE = events_df["type"].dropna().iloc[0]

filtered_events_type = [e for e in events if e.get("type") == FILTER_TYPE]
filtered_graph_type = build_global_context_graph(filtered_events_type)

print("Type sélectionné :", FILTER_TYPE)
print("Nombre d'événements :", len(filtered_events_type))

if filtered_graph_type.number_of_nodes() > 0:
    plot_graph(filtered_graph_type, title=f"Graphe global filtré · type = {FILTER_TYPE}")

Type sélectionné : Thing/Abstract/Event/Win
Nombre d'événements : 45


In [44]:
def get_neighbors_table(G, node_id):
    if node_id not in G:
        return pd.DataFrame()

    rows = []

    for neighbor in G.successors(node_id):
        edge_data = G.get_edge_data(node_id, neighbor)
        rows.append({
            "direction": "out",
            "from": node_id,
            "to": neighbor,
            "edge_type": edge_data.get("edge_type"),
            "neighbor_label": G.nodes[neighbor].get("label"),
            "neighbor_type": G.nodes[neighbor].get("node_type"),
            "neighbor_main_label": G.nodes[neighbor].get("main_label"),
        })

    for neighbor in G.predecessors(node_id):
        edge_data = G.get_edge_data(neighbor, node_id)
        rows.append({
            "direction": "in",
            "from": neighbor,
            "to": node_id,
            "edge_type": edge_data.get("edge_type"),
            "neighbor_label": G.nodes[neighbor].get("label"),
            "neighbor_type": G.nodes[neighbor].get("node_type"),
            "neighbor_main_label": G.nodes[neighbor].get("main_label"),
        })

    return pd.DataFrame(rows)

In [45]:
some_node = list(G_global.nodes())[0]
neighbors_df = get_neighbors_table(G_global, some_node)

print("Nœud sélectionné :", some_node)
display(neighbors_df)

Nœud sélectionné : event::698b35e78f03efde04a87998


,direction,from,to,edge_type,neighbor_label,neighbor_type,neighbor_main_label
0,out,event::698b35e78f03efde04a87998,474738074,sourceNode,remporter,semantic_node,Thing/Abstract/Event/Win
1,out,event::698b35e78f03efde04a87998,2080660730,contains,fauteuil,semantic_node,Thing/Concrete/Inanimate
2,out,event::698b35e78f03efde04a87998,1212458420,contains,15et 22mars,semantic_node,Thing/Abstract/Time


In [46]:
events_df[["event_id", "type", "domain", "taskId", "sourceNodeId", "n_nodes", "n_edges"]].head(20)

,event_id,type,domain,taskId,sourceNodeId,n_nodes,n_edges
0,698b35e78f03efde04a87998,Thing/Abstract/Event/Win,NaN,1a076139-ea5e-40e2-8d82-01a1f28b8181,474738074,3,2
1,698b35e98f03efde04a8799a,Thing/Abstract/Event/Transfer/TransferOfUnbias...,Thing/Abstract/Event/Spying,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,1667751063,3,2
2,698b35e98f03efde04a8799b,Thing/Abstract/Event/Transfer/TransferOfUnbias...,Thing/Abstract/Event/Spying,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,562551427,3,2
3,698b35e98f03efde04a8799c,Thing/Abstract/Event/Build,Thing/Abstract/Event/Spying,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,397183198,1,0
4,698b35e98f03efde04a8799d,Thing/Abstract/Event/Addition,Thing/Abstract/Event/Spying,9c1f0cb5-58ed-44db-b348-1bb5e5cc3c75,722640268,3,2
5,698b35ea8f03efde04a8799f,Thing/Abstract/Event/Hold,NaN,18bcd492-375b-4564-a723-7cba604b63f1,1413780907,6,5
6,698b35ea8f03efde04a879a0,Thing/Abstract/Event/Communication,NaN,18bcd492-375b-4564-a723-7cba604b63f1,56638349,2,1
7,698b35ea8f03efde04a879a1,Thing/Abstract/Event/Confess,NaN,18bcd492-375b-4564-a723-7cba604b63f1,228831100,4,3
8,698b35ea8f03efde04a879a2,Thing/Abstract/Event/Own,NaN,18bcd492-375b-4564-a723-7cba604b63f1,959760768,1,0
9,698b35ea8f03efde04a879a3,Thing/Abstract/Event/Own,NaN,18bcd492-375b-4564-a723-7cba604b63f1,398853882,1,0


In [47]:
nodes_df[["event_id", "node_id", "form", "main_label"]].head(20)

,event_id,node_id,form,main_label
0,698b35e78f03efde04a87998,474738074,remporter,Thing/Abstract/Event/Win
1,698b35e78f03efde04a87998,2080660730,fauteuil,Thing/Concrete/Inanimate
2,698b35e78f03efde04a87998,1212458420,15et 22mars,Thing/Abstract/Time
3,698b35e98f03efde04a8799a,1667751063,publiés,Thing/Abstract/Event/Transfer/TransferOfUnbias...
4,698b35e98f03efde04a8799a,133445445,2026-02-10T13:42:59.932651238,Thing/Abstract/Time
5,698b35e98f03efde04a8799a,1664342503,e-mails,Thing/Concrete/Inanimate/Product/Communication...
6,698b35e98f03efde04a8799b,562551427,témoignent,Thing/Abstract/Event/Transfer/TransferOfUnbias...
7,698b35e98f03efde04a8799b,1667751063,publiés,Thing/Abstract/Event/Transfer/TransferOfUnbias...
8,698b35e98f03efde04a8799b,310862638,servi,Thing
9,698b35e98f03efde04a8799c,397183198,montages,Thing/Abstract/Event/Build


In [48]:
nodes_df[["event_id", "node_id", "form", "main_label"]].head(20)

,event_id,node_id,form,main_label
0,698b35e78f03efde04a87998,474738074,remporter,Thing/Abstract/Event/Win
1,698b35e78f03efde04a87998,2080660730,fauteuil,Thing/Concrete/Inanimate
2,698b35e78f03efde04a87998,1212458420,15et 22mars,Thing/Abstract/Time
3,698b35e98f03efde04a8799a,1667751063,publiés,Thing/Abstract/Event/Transfer/TransferOfUnbias...
4,698b35e98f03efde04a8799a,133445445,2026-02-10T13:42:59.932651238,Thing/Abstract/Time
5,698b35e98f03efde04a8799a,1664342503,e-mails,Thing/Concrete/Inanimate/Product/Communication...
6,698b35e98f03efde04a8799b,562551427,témoignent,Thing/Abstract/Event/Transfer/TransferOfUnbias...
7,698b35e98f03efde04a8799b,1667751063,publiés,Thing/Abstract/Event/Transfer/TransferOfUnbias...
8,698b35e98f03efde04a8799b,310862638,servi,Thing
9,698b35e98f03efde04a8799c,397183198,montages,Thing/Abstract/Event/Build


In [49]:
type_stats = (
    events_df.groupby("type", dropna=False)
    .agg(
        n_events=("event_id", "count"),
        avg_nodes=("n_nodes", "mean"),
        avg_edges=("n_edges", "mean"),
    )
    .sort_values("n_events", ascending=False)
)

display(type_stats.head(20))

,n_events,avg_nodes,avg_edges
type,,,
Thing/Abstract/Event/Own,1125,1.230222,0.235556
Thing/Abstract/Event/Transfer/TransferOfUnbiasedInformation,1043,3.729626,2.790029
Thing/Abstract/Event/Addition,964,3.371369,2.376556
Thing/Abstract/Event/Unknown,844,3.271327,2.309242
Thing/Abstract/Event,372,1.311828,0.322581
Thing/Abstract/Event/Transfer,308,2.990260,2.045455
Thing/Abstract/Event/Move,293,3.457338,2.535836
Thing/Abstract/Event/Partof,218,3.000000,2.000000
Thing/Abstract/Event/Give,215,3.590698,2.823256


In [50]:
domain_stats = (
    events_df.fillna({"domain": "NA"})
    .groupby("domain")
    .agg(
        n_events=("event_id", "count"),
        avg_nodes=("n_nodes", "mean"),
        avg_edges=("n_edges", "mean"),
    )
    .sort_values("n_events", ascending=False)
)

display(domain_stats.head(20))

,n_events,avg_nodes,avg_edges
domain,,,
NA,8345,2.999161,2.055482
Thing/Abstract/Organization/Operator/Military,884,2.918552,1.975113
Thing/Abstract/Organization/Operator/Civil,641,2.967239,2.028081
Thing/Abstract/Organization/Operator/Energy,350,2.674286,1.688571
Thing/Abstract/Event/DataLeak,279,3.186380,2.197133
Thing/Abstract/Event/Spying,263,3.083650,2.163498
Thing/Abstract/Domain/Weather,229,3.244541,2.296943
Thing/Abstract/Organization/Operator/Communication,203,3.000000,2.024631
Thing/Abstract/Event/SocialMovement,178,2.955056,2.000000


In [51]:
task_stats = (
    events_df.groupby("taskId", dropna=False)
    .agg(
        n_events=("event_id", "count"),
        avg_nodes=("n_nodes", "mean"),
        avg_edges=("n_edges", "mean"),
    )
    .sort_values("n_events", ascending=False)
)

display(task_stats.head(20))

,n_events,avg_nodes,avg_edges
taskId,,,
fcb04a09-d91d-45e4-9884-a1ae0b44bf53,21,3.000000,2.142857
4b755bc0-a4de-4b56-a78b-bfd86df864a0,19,2.789474,1.842105
a0d0c287-31d2-486a-a948-b83259a37d79,19,3.052632,2.052632
9832a384-9209-4069-83d7-335fc51930b6,18,2.666667,1.666667
7fe2b78b-8f42-4158-b3bf-b5cb7708eb95,18,3.333333,2.388889
b3ea4f03-df9a-492d-b4a1-168489b5b790,17,2.117647,1.117647
fce6c3c0-6320-4fc7-8666-a4e33d6bc2b1,16,3.062500,2.125000
f1db104a-f248-49ab-bbd6-60c3a28ff7a9,16,3.125000,2.125000
2ead8c0f-6bc4-42dd-9f95-cc60b8710267,16,2.687500,1.812500
